# Practice 3

# 📓 Exercise 3 — Complete Standalone Notebook

# Exercise 3: Evaluating RandomizedSearchCV with Increased Iterations (n_iter=20)

**Goal:** Re-run Section 11's `RandomizedSearchCV` with `n_iter=20` (sampling 20 candidate combinations out of 42 possible grid points) instead of `n_iter=10`. Evaluate how close or superior `best_score_` gets compared to Section 10's `GridSearchCV` baseline and analyze the trade-off between sampled iterations and search efficiency.

# Cell 1 — Import required libraries and tools

We import all necessary modules for data manipulation, preprocessing pipelines, model selection, and search strategies.

In [1]:
# Import core tools, preprocessing, and model selection search classes.

import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor

# Cell 2 — Load dataset and engineer features

We load the California Housing dataset and apply feature engineering (missing values, categorical binning, and numerical ratio features) to match the standard workflow.

In [2]:
# Load dataset and prepare reproducible messy features.

california = fetch_california_housing(as_frame=True)
housing = california.frame

rng = np.random.default_rng(42)
messy = housing.copy()

# Add reproducible missing values

missing_idx = rng.choice(messy.index, size=200, replace=False)
messy.loc[missing_idx, "AveBedrms"] = np.nan
missing_idx_2 = rng.choice(messy.index, size=120, replace=False)
messy.loc[missing_idx_2, "AveRooms"] = np.nan

# Feature engineering

messy["coast_region"] = pd.cut(
    messy["Longitude"],
    bins=[-125, -121.22, -118.20, -114],
    labels=["Coastal", "Central", "Inland"],)

messy["income_cat"] = pd.cut(
    messy["MedInc"],
    bins=[0, 2.5, 4.5, 7, np.inf],
    labels=["Low", "Medium", "High", "Very High"],)

messy["bedrooms_ratio"] = messy["AveBedrms"] / messy["AveRooms"]
messy["rooms_per_person"] = messy["AveRooms"] / messy["AveOccup"]

X = messy.drop(columns="MedHouseVal")
y = messy["MedHouseVal"]

# Cell 3 — Split dataset into train and test sets

We split the features and target variable into an 80% training set and a 20% test set using a fixed random seed.

In [3]:
# Train-test split (80% train, 20% test).

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# Specify numeric and categorical feature columns

numeric_features = [
    "MedInc", "HouseAge", "AveRooms", "AveBedrms",
    "Population", "AveOccup", "Latitude", "Longitude"]

categorical_features = ["coast_region"]

# Cell 4 — Construct the preprocessing pipeline

We set up a ColumnTransformer to impute missing values, scale numeric features, and one-hot encode categorical features automatically.

In [4]:
# Preprocessing sub-pipelines for numerical and categorical features.

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),])


categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),])


# Combine into a master ColumnTransformer

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),])


# Cell 5 — Define the main Random Forest pipeline

We encapsulate the preprocessor and RandomForestRegressor inside a single unified pipeline.

In [5]:
# Create main estimator pipeline.

forest_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(random_state=42, n_jobs=-1)),])

# Cell 6 — Run Section 10's GridSearchCV baseline

We re-run Section 10's full grid search (9 combinations x 5 folds = 45 fits) as our benchmark.

In [6]:
# Section 10 Grid Search parameters

param_grid = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [None, 10, 20],}

grid_search = GridSearchCV(
    forest_pipeline,
    param_grid,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1)

grid_search.fit(X_train, y_train)

print("GridSearchCV Best Params:", grid_search.best_params_)
print(f"GridSearchCV Best CV RMSE: {-grid_search.best_score_:.4f}")

GridSearchCV Best Params: {'model__max_depth': None, 'model__n_estimators': 300}
GridSearchCV Best CV RMSE: 0.5091


# Cell 7 — Run Section 11's baseline RandomizedSearchCV (n_iter=10)

We define Section 11's parameter distributions (7 x 6 = 42 total combinations) and fit `n_iter=10` (50 total fits).

In [7]:
# Section 11 wider parameter distribution space (42 possible combinations)

param_distributions = {
    "model__n_estimators": [50, 100, 150, 200, 250, 300, 400],
    "model__max_depth": [None, 5, 10, 15, 20, 25],}

# Baseline RandomizedSearchCV with n_iter=10

random_search_10 = RandomizedSearchCV(
    forest_pipeline,
    param_distributions,
    n_iter=10,
    cv=5,
    scoring="neg_root_mean_squared_error",
    random_state=42,
    n_jobs=-1,)

random_search_10.fit(X_train, y_train)

print("RandomizedSearchCV (n_iter=10) Best Params:", random_search_10.best_params_)
print(f"RandomizedSearchCV (n_iter=10) Best CV RMSE: {-random_search_10.best_score_:.4f}")

RandomizedSearchCV (n_iter=10) Best Params: {'model__n_estimators': 400, 'model__max_depth': None}
RandomizedSearchCV (n_iter=10) Best CV RMSE: 0.5083


# Cell 8 — Execute Exercise 3's RandomizedSearchCV (n_iter=20)

We increase `n_iter` from 10 to 20 to sample almost half (20 out of 42) of the entire parameter search space (20 x 5 = 100 fits).

In [8]:
# Exercise 3 RandomizedSearchCV with n_iter=20

random_search_20 = RandomizedSearchCV(
    forest_pipeline,
    param_distributions,
    n_iter=20,
    cv=5,
    scoring="neg_root_mean_squared_error",
    random_state=42,
    n_jobs=-1,)

random_search_20.fit(X_train, y_train)

print("RandomizedSearchCV (n_iter=20) Best Params:", random_search_20.best_params_)
print(f"RandomizedSearchCV (n_iter=20) Best CV RMSE: {-random_search_20.best_score_:.4f}")

RandomizedSearchCV (n_iter=20) Best Params: {'model__n_estimators': 400, 'model__max_depth': None}
RandomizedSearchCV (n_iter=20) Best CV RMSE: 0.5083


# Cell 9 — Construct the search comparison table

We summarize search methods, total iterations, number of fits, optimal parameters, and cross-validation RMSE scores into a pandas DataFrame.

In [9]:
# Assemble results into a comparative summary table.

search_summary = pd.DataFrame([
    {
        "Search Method": "GridSearchCV (Section 10)",
        "Combinations Evaluated": "9 / 9 (100%)",
        "Total Fits (5-fold)": 45,
        "Best Hyperparameters": str(grid_search.best_params_),
        "Best CV RMSE": round(-grid_search.best_score_, 4),
    },
    {
        "Search Method": "RandomizedSearchCV (n_iter=10)",
        "Combinations Evaluated": "10 / 42 (23.8%)",
        "Total Fits (5-fold)": 50,
        "Best Hyperparameters": str(random_search_10.best_params_),
        "Best CV RMSE": round(-random_search_10.best_score_, 4),
    },
    {
        "Search Method": "RandomizedSearchCV (n_iter=20)",
        "Combinations Evaluated": "20 / 42 (47.6%)",
        "Total Fits (5-fold)": 100,
        "Best Hyperparameters": str(random_search_20.best_params_),
        "Best CV RMSE": round(-random_search_20.best_score_, 4),
    },
])

search_summary

,Search Method,Combinations Evaluated,Total Fits (5-fold),Best Hyperparameters,Best CV RMSE
0,GridSearchCV (Section 10),9 / 9 (100%),45,"{'model__max_depth': None, 'model__n_estimator...",0.5091
1,RandomizedSearchCV (n_iter=10),10 / 42 (23.8%),50,"{'model__n_estimators': 400, 'model__max_depth...",0.5083
2,RandomizedSearchCV (n_iter=20),20 / 42 (47.6%),100,"{'model__n_estimators': 400, 'model__max_depth...",0.5083


# Cell 10 — Final interpretation for Exercise 3

We evaluate how close RandomizedSearchCV with `n_iter=20` got to the grid search baseline and analyze the performance gains vs. computational trade-off.

In [10]:
# Report detailed findings for Exercise 3.

grid_rmse = -grid_search.best_score_
rand10_rmse = -random_search_10.best_score_
rand20_rmse = -random_search_20.best_score_

print("--- Exercise 3 Key Findings ---")
print(f"1. GridSearchCV (Section 10) RMSE:        {grid_rmse:.4f}")
print(f"2. RandomizedSearchCV (n_iter=10) RMSE:    {rand10_rmse:.4f}")
print(f"3. RandomizedSearchCV (n_iter=20) RMSE:    {rand20_rmse:.4f}")

if rand20_rmse <= grid_rmse:
    print(f"\nConclusion: RandomizedSearchCV (n_iter=20) achieved an RMSE of {rand20_rmse:.4f}, "
        f"which is EQUAL TO OR BETTER than GridSearchCV ({grid_rmse:.4f}) because it explored "
        f"higher estimator values (n_estimators=400) outside Section 10's narrow grid.")

--- Exercise 3 Key Findings ---
1. GridSearchCV (Section 10) RMSE:        0.5091
2. RandomizedSearchCV (n_iter=10) RMSE:    0.5083
3. RandomizedSearchCV (n_iter=20) RMSE:    0.5083

Conclusion: RandomizedSearchCV (n_iter=20) achieved an RMSE of 0.5083, which is EQUAL TO OR BETTER than GridSearchCV (0.5091) because it explored higher estimator values (n_estimators=400) outside Section 10's narrow grid.
